# 3D reporter timelapse — 04c_reporter_background_distribution_qc

**Feeds:** Fig 5h

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# Reporter Background Distribution QC

This notebook asks one focused question before reporter thresholding:

**Is a per-image scalar background subtraction a reasonable correction for off-cyst reporter pixels after illumination correction?**

The goal is not to standardize by a background sigma. The goal is only to test whether a per-frame scalar shift removes most of the image-to-image off-cyst drift, or whether substantial residual width / shape differences remain afterward.


## Setup

This section loads the existing `04b` frame roster, the illumination fields, and helper functions for measuring off-cyst reporter-pixel distributions directly from the raw images and organoid masks.


In [ ]:
import json
import math
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile as tiff
from IPython.display import Markdown, display
from matplotlib.ticker import FuncFormatter

pd.set_option("display.max_columns", 200)
plt.rcParams["figure.dpi"] = 120


In [ ]:
import sys

cwd = Path.cwd().resolve()
root_candidates = [cwd] + list(cwd.parents[:3])
ROOT = None
for candidate in root_candidates:
    if (candidate / "data").exists() and (candidate / "results").exists():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Could not locate project root from current working directory.")

qc_dir = ROOT / "scripts" / "qc"
if str(qc_dir) not in sys.path:
    sys.path.insert(0, str(qc_dir))

from notebook_display_helpers import (
    TIME_DISPLAY_OFFSET_HOURS,
    display_time_df as _display_time_df,
    display_time_hours as _display_time_hours,
    format_display_hours as _format_display_hours,
    set_display_time_axis as _set_display_time_axis,
)
from notebook_invariant_helpers import (
    assert_background_cutoff_application,
)

DATASET_DIR = ROOT / "data/raw/20260128_BMP4-reporter_LPM-organoids/d2-d5"
MASK_ROOT = ROOT / "results/ilastik/organoid_masks/full_dataset_v1"
ACQUISITION_SUMMARY_PATH = ROOT / "results/qc/acquisition_qc_summary.json"
ILLUMINATION_FIELD_PATH = ROOT / "results/qc/04_masked_illumination_fields.npz"
BACKGROUND_TRACE_PATH = ROOT / "results/tables/04b_background_trace_by_frame.tsv"
BACKGROUND_EXCLUSION_WINDOW_PATH = ROOT / "results/tables/04b_background_exclusion_windows.tsv"

FIGURE_DIR = ROOT / "results/figures/04c"
TABLE_DIR = ROOT / "results/tables"
EXECUTED_NOTEBOOK_DIR = ROOT / "results/executed_notebooks"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
EXECUTED_NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

acquisition_summary = json.loads(ACQUISITION_SUMMARY_PATH.read_text())
interval_minutes = float(acquisition_summary["interval_minutes"])
payload = np.load(ILLUMINATION_FIELD_PATH)
illumination_fields = {
    "RFP": payload["rfp_field"].astype(np.float32),
    "YFP": payload["yfp_field"].astype(np.float32),
}
background_trace_df = pd.read_csv(BACKGROUND_TRACE_PATH, sep="\t", low_memory=False)
background_exclusion_windows = (
    pd.read_csv(BACKGROUND_EXCLUSION_WINDOW_PATH, sep="\t")
    if BACKGROUND_EXCLUSION_WINDOW_PATH.exists()
    else pd.DataFrame(columns=["position_label", "background_qc_cut_start_time_index"])
)

REPORTER_ORDER = ["RFP", "YFP"]
REPORTER_DISPLAY = {"RFP": "FOXF1-RFP", "YFP": "BMP4-YFP"}
REPORTER_COLORS = {"RFP": "tab:red", "YFP": "goldenrod"}
POSITION_RE = re.compile(r"Pos(?P<position_index>\d+)$")
QUANTILE_LEVELS = np.array([0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99], dtype=float)
QUANTILE_LABELS = [f"q{int(round(level * 100)):02d}" for level in QUANTILE_LEVELS]
CENTERED_QUANTILE_LABELS = [f"centered_{label}" for label in QUANTILE_LABELS]
QUANTILE_PERCENTILES = QUANTILE_LEVELS * 100.0
FRAME_SAMPLE_COUNT = 256
TOP_OUTLIER_COUNT = 10
QUANTILE_DISPLAY_SAMPLE_PER_REPORTER = 300
RNG = np.random.default_rng(0)

print("Project root:", ROOT)
print("Background-trace rows from 04b:", len(background_trace_df))
print("Background exclusion windows from 04b:", len(background_exclusion_windows))
print("Unique reporter-frame rows before applying 04b cut windows:", background_trace_df[["position_label", "reporter", "time_index"]].drop_duplicates().shape[0])
print("Interval minutes:", interval_minutes)


In [ ]:
def position_index_from_label(position_label: str) -> int:
    match = POSITION_RE.fullmatch(position_label)
    if not match:
        raise ValueError(f"Unexpected position label: {position_label}")
    return int(match.group("position_index"))


def display_time_hours(values):
    return _display_time_hours(values, offset_hours=TIME_DISPLAY_OFFSET_HOURS)


def display_time_df(df):
    return _display_time_df(df, offset_hours=TIME_DISPLAY_OFFSET_HOURS)


def format_display_hours(value: float, decimals: int = 1) -> str:
    return _format_display_hours(value, decimals=decimals, offset_hours=TIME_DISPLAY_OFFSET_HOURS)


def format_timepoint_label(time_index: int | float, time_hours: float | None = None, decimals: int = 1) -> str:
    if time_hours is None:
        time_hours = float(time_index) * interval_minutes / 60.0
    return f"t{int(time_index)} ({format_display_hours(float(time_hours), decimals)})"


def set_display_time_axis(ax, axis: str = "x", crowded: bool = False) -> None:
    _set_display_time_axis(ax, axis=axis, crowded=crowded, offset_hours=TIME_DISPLAY_OFFSET_HOURS)


def save_figure(fig, stem: str) -> dict[str, Path]:
    outputs = {}
    for suffix in [".png", ".pdf", ".svg"]:
        path = FIGURE_DIR / f"{stem}{suffix}"
        fig.savefig(path, dpi=180 if suffix == ".png" else None, bbox_inches="tight")
        outputs[suffix] = path
    return outputs


def raw_frame_path(position_label: str, channel_index: int, time_index: int) -> Path:
    position_index = position_index_from_label(position_label)
    return (
        DATASET_DIR
        / position_label
        / f"img_channel{channel_index:03d}_position{position_index:03d}_time{int(time_index):09d}_z000.tif"
    )


def mask_frame_path(position_label: str, time_index: int) -> Path:
    position_index = position_index_from_label(position_label)
    return (
        MASK_ROOT
        / position_label
        / f"img_channel000_position{position_index:03d}_time{int(time_index):09d}_z000_mask.tiff"
    )


def load_mask(position_label: str, time_index: int) -> np.ndarray:
    path = mask_frame_path(position_label, time_index)
    if not path.exists():
        raise FileNotFoundError(f"Missing mask: {path}")
    return tiff.imread(path).astype(bool)


def load_reporter_after_illumination(position_label: str, reporter: str, time_index: int) -> np.ndarray:
    channel_index = 1 if reporter == "RFP" else 2
    image = tiff.imread(raw_frame_path(position_label, channel_index, time_index)).astype(np.float32)
    return image / np.clip(illumination_fields[reporter], 1e-6, None)


def crop_bounds_from_mask(mask: np.ndarray, pad: int = 28) -> tuple[int, int, int, int]:
    coords = np.argwhere(mask)
    if coords.size == 0:
        return 0, mask.shape[0], 0, mask.shape[1]
    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0) + 1
    y0 = max(0, int(y0) - pad)
    x0 = max(0, int(x0) - pad)
    y1 = min(mask.shape[0], int(y1) + pad)
    x1 = min(mask.shape[1], int(x1) + pad)
    return y0, y1, x0, x1


def pooled_image_limits(images: list[np.ndarray], low_q: float = 0.01, high_q: float = 0.995) -> tuple[float, float]:
    pooled = []
    for image in images:
        finite = np.asarray(image, dtype=float)
        finite = finite[np.isfinite(finite)]
        if finite.size:
            pooled.append(finite)
    if not pooled:
        return 0.0, 1.0
    values = np.concatenate(pooled)
    vmin = float(np.quantile(values, low_q))
    vmax = float(np.quantile(values, high_q))
    if not np.isfinite(vmin):
        vmin = float(np.nanmin(values))
    if not np.isfinite(vmax):
        vmax = float(np.nanmax(values))
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        vmax = vmin + 1.0
    return vmin, vmax


def sample_without_replacement(values: np.ndarray, max_count: int) -> np.ndarray:
    values = np.asarray(values)
    if values.size <= max_count:
        return values.astype(np.float32, copy=False)
    indices = RNG.choice(values.size, size=max_count, replace=False)
    return values[indices].astype(np.float32, copy=False)


def centered_hist_limits(values: np.ndarray, lower_q: float = 0.01, upper_q: float = 0.99) -> tuple[float, float]:
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        return -1.0, 1.0
    lo = float(np.quantile(finite, lower_q))
    hi = float(np.quantile(finite, upper_q))
    bound = max(abs(lo), abs(hi))
    if not np.isfinite(bound) or bound <= 0:
        bound = 1.0
    return -bound, bound


## Measure Off-Cyst Pixel Distributions

Starting from the `04b` reporter-frame roster, this section first removes any frames at or after the explicit `04b` background-QC cut start for that position. It then re-loads the raw reporter image, applies the illumination correction, takes the whole off-cyst pixels, and records frame-level quantiles before and after subtracting that frame's off-cyst median.


In [ ]:
if not background_exclusion_windows.empty:
    cut_start_lookup = (
        background_exclusion_windows[["position_label", "background_qc_cut_start_time_index"]]
        .dropna(subset=["background_qc_cut_start_time_index"])
        .drop_duplicates("position_label")
        .set_index("position_label")["background_qc_cut_start_time_index"]
        .astype(float)
        .to_dict()
    )
else:
    cut_start_lookup = {}

background_trace_df["cut_start_time_index"] = background_trace_df["position_label"].map(cut_start_lookup)
background_trace_df_pre_cut = background_trace_df.copy()
pre_cut_reporter_frame_count = background_trace_df[["position_label", "reporter", "time_index"]].drop_duplicates().shape[0]
background_trace_df = background_trace_df.loc[
    background_trace_df["cut_start_time_index"].isna()
    | (background_trace_df["time_index"].astype(float) < background_trace_df["cut_start_time_index"].astype(float))
].copy()
post_cut_reporter_frame_count = background_trace_df[["position_label", "reporter", "time_index"]].drop_duplicates().shape[0]
assert_background_cutoff_application(
    background_trace_df_pre_cut,
    background_trace_df,
    cut_start_lookup,
    key_columns=("position_label", "reporter", "time_index"),
    position_column="position_label",
    time_column="time_index",
    context="04c retained-frame filtering after 04b cut windows",
)

unique_frames = (
    background_trace_df[["position_label", "time_index", "time_hours"]]
    .drop_duplicates()
    .sort_values(["position_label", "time_index"])
    .reset_index(drop=True)
)

distribution_rows = []
pooled_samples_raw = {reporter: [] for reporter in REPORTER_ORDER}
pooled_samples_centered = {reporter: [] for reporter in REPORTER_ORDER}

for frame_counter, frame_row in enumerate(unique_frames.itertuples(index=False), start=1):
    position_label = str(frame_row.position_label)
    time_index = int(frame_row.time_index)
    time_hours = float(frame_row.time_hours)
    mask = load_mask(position_label, time_index)
    off_cyst_mask = ~mask
    if not np.any(off_cyst_mask):
        continue

    if frame_counter % 500 == 0:
        print(f"Processed {frame_counter}/{len(unique_frames)} retained frames")

    for reporter in REPORTER_ORDER:
        signal = load_reporter_after_illumination(position_label, reporter, time_index).astype(np.float32)
        values = signal[off_cyst_mask].astype(np.float32)
        if values.size == 0:
            continue
        median_value = float(np.median(values))
        quantiles = np.quantile(values, QUANTILE_LEVELS)
        centered_quantiles = quantiles - median_value
        q_lookup = dict(zip(QUANTILE_LABELS, quantiles))
        centered_lookup = dict(zip(CENTERED_QUANTILE_LABELS, centered_quantiles))
        q05 = float(q_lookup["q05"])
        q25 = float(q_lookup["q25"])
        q50 = float(q_lookup["q50"])
        q75 = float(q_lookup["q75"])
        q95 = float(q_lookup["q95"])
        row = {
            "position_label": position_label,
            "time_index": time_index,
            "time_hours": time_hours,
            "reporter": reporter,
            "off_cyst_pixel_count": int(values.size),
            "background_median": median_value,
            "background_iqr": float(q75 - q25),
            "background_q95_minus_q05": float(q95 - q05),
            "background_mad": float(np.median(np.abs(values - median_value))),
        }
        row.update({label: float(value) for label, value in q_lookup.items()})
        row.update({label: float(value) for label, value in centered_lookup.items()})
        distribution_rows.append(row)

        pooled_samples_raw[reporter].append(sample_without_replacement(values, FRAME_SAMPLE_COUNT))
        pooled_samples_centered[reporter].append(sample_without_replacement(values - median_value, FRAME_SAMPLE_COUNT))

distribution_df = pd.DataFrame(distribution_rows).sort_values(["reporter", "position_label", "time_index"]).reset_index(drop=True)
cached_background = background_trace_df.rename(columns={"background_value": "cached_background_median"})
distribution_df = distribution_df.merge(
    cached_background,
    on=["position_label", "reporter", "time_index", "time_hours"],
    how="left",
)
distribution_df["background_recompute_delta"] = distribution_df["background_median"] - distribution_df["cached_background_median"]

reference_curve_rows = []
summary_rows = []
example_rows = []
top_outlier_rows = []
q_matrix_columns = QUANTILE_LABELS
centered_matrix_columns = [label for label in CENTERED_QUANTILE_LABELS if label != "centered_q50"]
core_q_matrix_columns = ["q10", "q25", "q50", "q75"]
core_centered_matrix_columns = [
    "centered_q10",
    "centered_q25",
    "centered_q50",
    "centered_q75",
]

for reporter in REPORTER_ORDER:
    reporter_mask = distribution_df["reporter"] == reporter
    sub = distribution_df.loc[reporter_mask].copy()
    raw_reference = sub[q_matrix_columns].median(axis=0)
    centered_reference = sub[CENTERED_QUANTILE_LABELS].median(axis=0)

    raw_residual = sub[q_matrix_columns].to_numpy(dtype=float) - raw_reference.to_numpy(dtype=float)
    centered_residual = sub[centered_matrix_columns].to_numpy(dtype=float) - centered_reference[centered_matrix_columns].to_numpy(dtype=float)
    raw_core_reference = sub[core_q_matrix_columns].median(axis=0)
    centered_core_reference = sub[core_centered_matrix_columns].median(axis=0)
    raw_core_residual = sub[core_q_matrix_columns].to_numpy(dtype=float) - raw_core_reference.to_numpy(dtype=float)
    centered_core_residual = sub[core_centered_matrix_columns].to_numpy(dtype=float) - centered_core_reference.to_numpy(dtype=float)

    distribution_df.loc[reporter_mask, "raw_quantile_rmse"] = np.sqrt(np.mean(raw_residual**2, axis=1))
    distribution_df.loc[reporter_mask, "centered_quantile_rmse"] = np.sqrt(np.mean(centered_residual**2, axis=1))
    distribution_df.loc[reporter_mask, "centered_quantile_max_abs"] = np.max(np.abs(centered_residual), axis=1)
    distribution_df.loc[reporter_mask, "raw_core_quantile_rmse"] = np.sqrt(np.mean(raw_core_residual**2, axis=1))
    distribution_df.loc[reporter_mask, "centered_core_quantile_rmse"] = np.sqrt(np.mean(centered_core_residual**2, axis=1))
    distribution_df.loc[reporter_mask, "centered_core_quantile_max_abs"] = np.max(np.abs(centered_core_residual), axis=1)

    for percentile, raw_value, centered_value in zip(QUANTILE_PERCENTILES, raw_reference.tolist(), centered_reference.tolist()):
        reference_curve_rows.append(
            {
                "reporter": reporter,
                "percentile": float(percentile),
                "raw_reference_quantile": float(raw_value),
                "centered_reference_quantile": float(centered_value),
            }
        )

    summary_rows.append(
        {
            "reporter": reporter,
            "frame_count": int(sub.shape[0]),
            "background_median_p05": float(sub["background_median"].quantile(0.05)),
            "background_median_p50": float(sub["background_median"].quantile(0.50)),
            "background_median_p95": float(sub["background_median"].quantile(0.95)),
            "background_iqr_p05": float(sub["background_iqr"].quantile(0.05)),
            "background_iqr_p50": float(sub["background_iqr"].quantile(0.50)),
            "background_iqr_p95": float(sub["background_iqr"].quantile(0.95)),
            "raw_quantile_rmse_p50": float(distribution_df.loc[reporter_mask, "raw_quantile_rmse"].quantile(0.50)),
            "raw_quantile_rmse_p95": float(distribution_df.loc[reporter_mask, "raw_quantile_rmse"].quantile(0.95)),
            "centered_quantile_rmse_p50": float(distribution_df.loc[reporter_mask, "centered_quantile_rmse"].quantile(0.50)),
            "centered_quantile_rmse_p95": float(distribution_df.loc[reporter_mask, "centered_quantile_rmse"].quantile(0.95)),
            "centered_quantile_max_abs_p95": float(distribution_df.loc[reporter_mask, "centered_quantile_max_abs"].quantile(0.95)),
            "centered_core_quantile_rmse_p50": float(distribution_df.loc[reporter_mask, "centered_core_quantile_rmse"].quantile(0.50)),
            "centered_core_quantile_rmse_p95": float(distribution_df.loc[reporter_mask, "centered_core_quantile_rmse"].quantile(0.95)),
        }
    )

    ranked = distribution_df.loc[reporter_mask].sort_values(
        ["centered_core_quantile_rmse", "position_label", "time_index"]
    ).reset_index(drop=True)
    if ranked.empty:
        continue
    top_outliers = (
        ranked.tail(min(TOP_OUTLIER_COUNT, len(ranked)))
        .sort_values(["centered_core_quantile_rmse", "position_label", "time_index"], ascending=[False, True, True])
        .reset_index(drop=True)
    )
    top_example_row = top_outliers.iloc[0]
    example_specs = [
        ("representative", ranked.iloc[int(round((len(ranked) - 1) * 0.50))]),
        ("top-outlier", top_example_row),
    ]
    seen_examples = set()
    selected = []
    for order_label, row in example_specs:
        key = (str(row["position_label"]), int(row["time_index"]))
        if key in seen_examples:
            continue
        seen_examples.add(key)
        selected.append(
            {
                "reporter": reporter,
                "example_kind": order_label,
                "position_label": str(row["position_label"]),
                "time_index": int(row["time_index"]),
                "time_hours": float(row["time_hours"]),
                "background_median": float(row["background_median"]),
                "background_iqr": float(row["background_iqr"]),
                "centered_quantile_rmse": float(row["centered_quantile_rmse"]),
                "centered_quantile_max_abs": float(row["centered_quantile_max_abs"]),
                "centered_core_quantile_rmse": float(row["centered_core_quantile_rmse"]),
                "centered_core_quantile_max_abs": float(row["centered_core_quantile_max_abs"]),
            }
        )
    example_rows.extend(selected)
    for outlier_rank, row in enumerate(top_outliers.itertuples(index=False), start=1):
        top_outlier_rows.append(
            {
                "reporter": reporter,
                "outlier_rank": int(outlier_rank),
                "position_label": str(row.position_label),
                "time_index": int(row.time_index),
                "time_hours": float(row.time_hours),
                "background_median": float(row.background_median),
                "background_iqr": float(row.background_iqr),
                "centered_quantile_rmse": float(row.centered_quantile_rmse),
                "centered_quantile_max_abs": float(row.centered_quantile_max_abs),
                "centered_core_quantile_rmse": float(row.centered_core_quantile_rmse),
                "centered_core_quantile_max_abs": float(row.centered_core_quantile_max_abs),
            }
        )

summary_df = pd.DataFrame(summary_rows).sort_values("reporter").reset_index(drop=True)
reference_curve_df = pd.DataFrame(reference_curve_rows).sort_values(["reporter", "percentile"]).reset_index(drop=True)
example_df = pd.DataFrame(example_rows).sort_values(["reporter", "example_kind"]).reset_index(drop=True)
top_outlier_df = pd.DataFrame(top_outlier_rows).sort_values(["reporter", "outlier_rank"]).reset_index(drop=True)
quantile_display_sample_df = (
    pd.concat(
        [
            distribution_df.loc[distribution_df["reporter"] == reporter]
            .sample(
                n=min(QUANTILE_DISPLAY_SAMPLE_PER_REPORTER, int((distribution_df["reporter"] == reporter).sum())),
                random_state=0,
            )[
                [
                    "reporter",
                    "position_label",
                    "time_index",
                    "time_hours",
                    "background_median",
                    "off_cyst_pixel_count",
                    "raw_core_quantile_rmse",
                    "centered_core_quantile_rmse",
                    *QUANTILE_LABELS,
                    *CENTERED_QUANTILE_LABELS,
                ]
            ]
            .assign(display_sample_kind="quantile-cloud")
            for reporter in REPORTER_ORDER
            if (distribution_df["reporter"] == reporter).any()
        ],
        ignore_index=True,
    )
    .sort_values(["reporter", "position_label", "time_index"])
    .reset_index(drop=True)
)

pooled_reference_raw = {
    reporter: np.concatenate(samples).astype(np.float32) if samples else np.array([], dtype=np.float32)
    for reporter, samples in pooled_samples_raw.items()
}
pooled_reference_centered = {
    reporter: np.concatenate(samples).astype(np.float32) if samples else np.array([], dtype=np.float32)
    for reporter, samples in pooled_samples_centered.items()
}

distribution_table_path = TABLE_DIR / "04c_background_distribution_by_frame.tsv"
summary_table_path = TABLE_DIR / "04c_background_distribution_summary.tsv"
reference_table_path = TABLE_DIR / "04c_background_distribution_reference_quantiles.tsv"
example_table_path = TABLE_DIR / "04c_background_distribution_examples.tsv"
top_outlier_table_path = TABLE_DIR / "04c_background_distribution_top_outliers.tsv"
quantile_display_sample_table_path = TABLE_DIR / "04c_background_distribution_quantile_curve_display_sample.tsv"
distribution_df.to_csv(distribution_table_path, sep="\t", index=False)
summary_df.to_csv(summary_table_path, sep="\t", index=False)
reference_curve_df.to_csv(reference_table_path, sep="\t", index=False)
example_df.to_csv(example_table_path, sep="\t", index=False)
top_outlier_df.to_csv(top_outlier_table_path, sep="\t", index=False)
display_time_df(quantile_display_sample_df).to_csv(quantile_display_sample_table_path, sep="\t", index=False)

recompute_delta = distribution_df["background_recompute_delta"].abs()
display(
    Markdown(
        "\n".join(
            [
                "### Cached-vs-recomputed median sanity check",
                "",
                f"- reporter-frame rows before applying 04b cut windows: `{pre_cut_reporter_frame_count}`",
                f"- reporter-frame rows after applying 04b cut windows: `{post_cut_reporter_frame_count}`",
                f"- measured reporter-frame distributions: `{len(distribution_df)}`",
                f"- max absolute delta from cached 04b background median: `{recompute_delta.max():.6f}`",
                f"- median absolute delta from cached 04b background median: `{recompute_delta.median():.6f}`",
            ]
        )
    )
)
display(summary_df)
display(example_df)
display(top_outlier_df)
print("Wrote frame-level distribution diagnostics:", distribution_table_path)
print("Wrote summary table:", summary_table_path)
print("Wrote reference-curve table:", reference_table_path)
print("Wrote example-frame table:", example_table_path)
print("Wrote top-outlier table:", top_outlier_table_path)
print("Wrote quantile display-sample table:", quantile_display_sample_table_path)


## Quantile-Curve View Of Image-To-Image Background Distributions

Each thin line below is one image's off-cyst quantile curve. The faint cloud is still a display sample, but the highlighted curves are the **top 10 outliers across all retained frames** for that reporter, ranked by the median-centered `q10-q75` RMSE. The shaded band is the central 10-90% span across all retained frames for that reporter.

Selection manifests written by this notebook:

- faint-cloud display sample: `results/tables/04c_background_distribution_quantile_curve_display_sample.tsv`
- highlighted outliers: `results/tables/04c_background_distribution_top_outliers.tsv`


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11), constrained_layout=True, sharex=True)
outlier_style_lookup = {
    "RFP": {
        "top10-outlier": dict(color="darkred", linestyle="--"),
    },
    "YFP": {
        "top10-outlier": dict(color="saddlebrown", linestyle="--"),
    },
}

for row_idx, reporter in enumerate(REPORTER_ORDER):
    color = REPORTER_COLORS[reporter]
    display_name = REPORTER_DISPLAY[reporter]
    sub = distribution_df.loc[distribution_df["reporter"] == reporter].copy()
    sampled = quantile_display_sample_df.loc[
        quantile_display_sample_df["reporter"] == reporter
    ].copy()

    for col_idx, (curve_columns, panel_title, ylabel) in enumerate(
        [
            (QUANTILE_LABELS, f"{display_name}: off-cyst distributions before subtraction", "Illumination-corrected\nintensity"),
            (CENTERED_QUANTILE_LABELS, f"{display_name}: off-cyst distributions after per-frame median subtraction", "Median-centered\nintensity"),
        ]
    ):
        ax = axes[row_idx, col_idx]
        for _, row in sampled.iterrows():
            ax.plot(
                QUANTILE_PERCENTILES,
                row[curve_columns].to_numpy(dtype=float),
                color=color,
                alpha=0.11,
                linewidth=1.15,
            )
        band_lo = sub[curve_columns].quantile(0.10)
        band_hi = sub[curve_columns].quantile(0.90)
        curve_median = sub[curve_columns].median()
        ax.fill_between(
            QUANTILE_PERCENTILES,
            band_lo.to_numpy(dtype=float),
            band_hi.to_numpy(dtype=float),
            color=color,
            alpha=0.18,
            linewidth=0,
        )
        ax.plot(
            QUANTILE_PERCENTILES,
            curve_median.to_numpy(dtype=float),
            color="black",
            linewidth=2.6,
            label="median across frames",
        )
        outlier_annotations = []
        outlier_rows = top_outlier_df.loc[top_outlier_df["reporter"] == reporter].copy()
        if not outlier_rows.empty:
            for example_row in outlier_rows.itertuples(index=False):
                matched = sub.loc[
                    (sub["position_label"] == example_row.position_label)
                    & (sub["time_index"] == example_row.time_index)
                ]
                if matched.empty:
                    continue
                style = outlier_style_lookup[reporter]["top10-outlier"]
                values = matched.iloc[0][curve_columns].to_numpy(dtype=float)
                linewidth = 3.2 if int(example_row.outlier_rank) == 1 else 2.0
                alpha = 0.98 if int(example_row.outlier_rank) <= 3 else 0.78
                ax.plot(
                    QUANTILE_PERCENTILES,
                    values,
                    color=style["color"],
                    linestyle=style["linestyle"],
                    linewidth=linewidth,
                    alpha=alpha,
                    zorder=5 + (TOP_OUTLIER_COUNT - int(example_row.outlier_rank)),
                )
                outlier_annotations.append(
                    {
                        "label": f"#{int(example_row.outlier_rank)} {example_row.position_label}, {format_timepoint_label(int(example_row.time_index), float(example_row.time_hours), 1)}",
                        "y": float(values[-1]),
                        "x": 99.0,
                        "color": style["color"],
                        "rmse": float(example_row.centered_core_quantile_rmse),
                    }
                )

        ax.set_title(panel_title, fontsize=11)
        ax.set_ylabel(ylabel)
        ax.set_xlim(1, 104)
        ax.grid(alpha=0.22, linewidth=0.5)
        if row_idx == 1:
            ax.set_xlabel("Percentile of off-cyst pixels within frame")
        if row_idx == 0 and col_idx == 1:
            ax.legend(loc="upper left", fontsize=8)
        if outlier_rows is not None and len(outlier_rows) and outlier_annotations:
            ymin, ymax = ax.get_ylim()
            y_span = max(ymax - ymin, 1.0)
            sorted_annotations = sorted(outlier_annotations, key=lambda item: item["y"], reverse=True)
            y_targets = np.linspace(ymax - 0.10 * y_span, ymin + 0.15 * y_span, len(sorted_annotations))
            for annotation, y_text in zip(sorted_annotations, y_targets):
                ax.annotate(
                    f"{annotation['label']}\nRMSE={annotation['rmse']:.1f}",
                    xy=(annotation["x"], annotation["y"]),
                    xytext=(100.35, y_text),
                    textcoords="data",
                    ha="left",
                    va="center",
                    fontsize=8.3,
                    color=annotation["color"],
                    bbox=dict(
                        facecolor="white",
                        alpha=0.9,
                        edgecolor=annotation["color"],
                        linewidth=0.7,
                        pad=1.8,
                    ),
                    arrowprops=dict(
                        arrowstyle="-",
                        color=annotation["color"],
                        linewidth=1.0,
                    ),
                    clip_on=False,
                    zorder=7,
                )

outputs = save_figure(fig, "background_distribution_quantile_curves")
display(fig)
plt.close(fig)
print("Wrote figure:", outputs[".png"])


## Representative And Outlier Reporter Images

These image panels show one representative frame and the top-ranked background outlier for each reporter under the median-centered `q10-q75` RMSE. Within each reporter row, the two images use the exact same display contrast so the residual background structure is visually comparable.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11.5, 9.2), constrained_layout=True)

for row_idx, reporter in enumerate(REPORTER_ORDER):
    display_name = REPORTER_DISPLAY[reporter]
    reporter_examples = (
        example_df.loc[
            (example_df["reporter"] == reporter)
            & (example_df["example_kind"].isin(["representative", "top-outlier"]))
        ]
        .copy()
    )
    order = {"representative": 0, "top-outlier": 1}
    reporter_examples["sort_key"] = reporter_examples["example_kind"].map(order)
    reporter_examples = reporter_examples.sort_values("sort_key").reset_index(drop=True)

    crops = []
    masks = []
    labels = []
    for example_row in reporter_examples.itertuples(index=False):
        mask = load_mask(str(example_row.position_label), int(example_row.time_index))
        image = load_reporter_after_illumination(str(example_row.position_label), reporter, int(example_row.time_index)).astype(float)
        y0, y1, x0, x1 = crop_bounds_from_mask(mask, pad=36)
        crops.append(image[y0:y1, x0:x1])
        masks.append(mask[y0:y1, x0:x1])
        labels.append(
            {
                "kind": str(example_row.example_kind),
                "position_label": str(example_row.position_label),
                "time_index": int(example_row.time_index),
                "time_hours": float(example_row.time_hours),
                "rmse": float(example_row.centered_core_quantile_rmse),
            }
        )

    vmin, vmax = pooled_image_limits(crops, low_q=0.01, high_q=0.995)
    for col_idx, (crop, crop_mask, label_row) in enumerate(zip(crops, masks, labels)):
        ax = axes[row_idx, col_idx]
        ax.imshow(crop, cmap="magma", vmin=vmin, vmax=vmax)
        ax.contour(crop_mask.astype(float), levels=[0.5], colors="cyan", linewidths=0.8)
        ax.set_xticks([])
        ax.set_yticks([])
        kind_title = "Representative" if label_row["kind"] == "representative" else "Top outlier"
        ax.set_title(
            "\n".join(
                [
                    f"{display_name} {kind_title}",
                    f"{label_row['position_label']} at {format_timepoint_label(label_row['time_index'], label_row['time_hours'], 1)}",
                    f"q10-q75 RMSE={label_row['rmse']:.1f}",
                ]
            ),
            fontsize=10,
        )
        if col_idx == 0:
            ax.set_ylabel(display_name, fontsize=11)

outputs = save_figure(fig, "background_distribution_image_examples")
display(fig)
plt.close(fig)
print("Wrote figure:", outputs[".png"])


## How Much Residual Shape Variation Remains After Median-Centering?

The histogram below compares each frame's `q10-q75` off-cyst quantile curve against the reporter-wide reference curve, before and after median subtraction. A large leftward shift means the per-frame scalar subtraction removes most frame-to-frame variability in the middle of the off-cyst distribution. A wide residual tail means shape differences still remain after centering.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8), constrained_layout=True, sharey=True)

for ax, reporter in zip(axes, REPORTER_ORDER):
    color = REPORTER_COLORS[reporter]
    display_name = REPORTER_DISPLAY[reporter]
    sub = distribution_df.loc[distribution_df["reporter"] == reporter].copy()
    raw_values = sub["raw_core_quantile_rmse"].to_numpy(dtype=float)
    centered_values = sub["centered_core_quantile_rmse"].to_numpy(dtype=float)
    lo = float(np.nanmin(np.concatenate([raw_values, centered_values])))
    hi = float(np.nanquantile(np.concatenate([raw_values, centered_values]), 0.995))
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        hi = lo + 1.0
    bins = np.linspace(lo, hi, 45)
    ax.hist(raw_values, bins=bins, color="0.75", alpha=0.85, label="before subtraction", edgecolor="white")
    ax.hist(centered_values, bins=bins, color=color, alpha=0.55, label="after median subtraction", edgecolor="white")
    ax.axvline(np.median(raw_values), color="0.35", linestyle="--", linewidth=1.8)
    ax.axvline(np.median(centered_values), color=color, linestyle="--", linewidth=2.0)
    ax.set_title(display_name, fontsize=11)
    ax.set_xlabel("Frame-to-reference q10-q75 RMSE")
    ax.grid(alpha=0.22, linewidth=0.5)
    ax.legend(loc="upper right", fontsize=8)
    ax.text(
        0.03,
        0.97,
        f"median before={np.median(raw_values):.1f}\nmedian after={np.median(centered_values):.1f}",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=8.3,
        bbox=dict(facecolor="white", alpha=0.82, edgecolor="none", pad=2.0),
    )
axes[0].set_ylabel("Frame count")

outputs = save_figure(fig, "background_distribution_rmse_before_after")
display(fig)
plt.close(fig)
print("Wrote figure:", outputs[".png"])


## Example Centered Off-Cyst Histograms

These panels show concrete frames after median subtraction. The black outline is the pooled centered off-cyst reference distribution for that reporter. If the per-image scalar subtraction were fully adequate, the colored frame histograms would closely match the black outline.


In [ ]:
def load_centered_off_cyst_values(position_label: str, reporter: str, time_index: int) -> np.ndarray:
    mask = load_mask(position_label, time_index)
    signal = load_reporter_after_illumination(position_label, reporter, time_index).astype(np.float32)
    values = signal[~mask].astype(np.float32)
    median_value = float(np.median(values))
    return values - median_value


max_examples = max(
    example_df.loc[example_df["reporter"] == reporter].shape[0]
    for reporter in REPORTER_ORDER
)
fig, axes = plt.subplots(2, max_examples, figsize=(5.4 * max_examples, 8.4), constrained_layout=True, sharex=False, sharey=False)
if max_examples == 1:
    axes = np.asarray(axes).reshape(2, 1)
for row_idx, reporter in enumerate(REPORTER_ORDER):
    color = REPORTER_COLORS[reporter]
    display_name = REPORTER_DISPLAY[reporter]
    ref_values = pooled_reference_centered[reporter].astype(float)
    xmin, xmax = centered_hist_limits(ref_values, 0.01, 0.99)
    bins = np.linspace(xmin, xmax, 90)
    reporter_examples = example_df.loc[example_df["reporter"] == reporter].copy()
    order = {"representative": 0, "top-outlier": 1}
    reporter_examples["sort_key"] = reporter_examples["example_kind"].map(order).fillna(99)
    reporter_examples = reporter_examples.sort_values("sort_key").reset_index(drop=True)
    for col_idx, example_row in enumerate(reporter_examples.itertuples(index=False)):
        ax = axes[row_idx, col_idx]
        frame_values = load_centered_off_cyst_values(
            position_label=str(example_row.position_label),
            reporter=reporter,
            time_index=int(example_row.time_index),
        ).astype(float)
        ax.hist(ref_values, bins=bins, density=True, histtype="step", color="black", linewidth=1.8, label="pooled centered reference")
        ax.hist(frame_values, bins=bins, density=True, histtype="stepfilled", color=color, alpha=0.42, edgecolor=color, linewidth=1.0, label="selected frame")
        ax.axvline(0.0, color="0.3", linestyle="--", linewidth=1.1)
        ax.set_xlim(xmin, xmax)
        ax.grid(alpha=0.18, linewidth=0.5)
        kind_title = "Representative" if str(example_row.example_kind) == "representative" else "Top outlier"
        ax.set_title(
            "\n".join(
                [
                    f"{display_name} {kind_title}",
                    f"{example_row.position_label} at {format_timepoint_label(int(example_row.time_index), float(example_row.time_hours), 2)}",
                    f"q10-q75 RMSE={float(example_row.centered_core_quantile_rmse):.1f}",
                ]
            ),
            fontsize=10,
        )
        if col_idx == 0:
            ax.set_ylabel("Density")
        if row_idx == 1:
            ax.set_xlabel("Median-centered off-cyst intensity")
        if row_idx == 0 and col_idx == max_examples - 1:
            ax.legend(loc="upper right", fontsize=8)
    for col_idx in range(len(reporter_examples), max_examples):
        axes[row_idx, col_idx].axis("off")

outputs = save_figure(fig, "background_distribution_centered_examples")
display(fig)
plt.close(fig)
print("Wrote figure:", outputs[".png"])


## Outputs

This notebook writes:

- `results/tables/04c_background_distribution_by_frame.tsv`
- `results/tables/04c_background_distribution_summary.tsv`
- `results/tables/04c_background_distribution_reference_quantiles.tsv`
- `results/tables/04c_background_distribution_examples.tsv`
- `results/figures/04c/background_distribution_quantile_curves.{png,pdf,svg}`
- `results/figures/04c/background_distribution_image_examples.{png,pdf,svg}`
- `results/figures/04c/background_distribution_rmse_before_after.{png,pdf,svg}`
- `results/figures/04c/background_distribution_centered_examples.{png,pdf,svg}`
